# Step10d — Dashboard Interactivo (HTML autónomo)

**Proyecto:** EFCI Ecuador (ENSO Flood Chronicity Index)
**Fase:** 10 — Productos, sección D: dashboard interactivo (Fase 0 §9)
**Autor / Author:** Jerson Achicanoy, Information Management and GIS Expert
**Fecha:** 2026-08-06

**Objetivo de este notebook:** generar un dashboard interactivo como
**un solo archivo HTML autónomo** (`outputs/dashboard/EFCI_Ecuador_Dashboard.html`)
— se abre con doble clic en cualquier navegador, sin servidor ni
conexión a una base de datos. Los datos van embebidos dentro del propio
archivo (JSON), y el filtrado ocurre en el navegador (JavaScript).

**decisión confirmada por el equipo (AskUserQuestion, 2026-08-06):** además
de este HTML autónomo, se genera una segunda versión editable dentro de
Jupyter (`Step10e_Ecuador_Dashboard_Interactive.ipynb`, con
`ipywidgets`) — "ambos, para después editar de ser necesario".

**Filtros implementados** (Fase 0 §9): Provincia, Categoría ENSO
(ICEN), Año, Mes, Variable de Impacto — afectan simultáneamente el
mapa (recoloreado por cantón), un gráfico de tendencia y una tabla de
ranking, todo del lado del cliente (sin recargar la página).

**Límite documentado, no oculto:** al ser un archivo estático, el
dashboard no se actualiza solo si el pipeline produce datos nuevos —
hay que volver a ejecutar este notebook para regenerarlo. No es un
dashboard "vivo" contra una base de datos.

**English:** Interactive dashboard as a self-contained HTML file: filters by province, ENSO category, year, month and impact variable, with map, annual chart and ranking table.

## 0. Configuración del entorno

**English:** Environment setup.

In [ ]:
# Standard library: system path handling
import sys
# JSON serialisation for the embedded data
import json
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper that creates folders when missing
from src.utilidades.io_utils import ensure_dir

# Tabular data
import pandas as pd
# Numerical arrays
import numpy as np
# Vector geodata
import geopandas as gpd

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration
config = load_config(CONFIG_PATH)
# Create the notebook logger
logger = get_logger("step10d_dashboard_html", config, log_dir=PROJECT_ROOT / "logs")
# Log the start of the notebook
logger.info("Fase 10d iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Output folder for the dashboard
dashboard_dir = ensure_dir(PROJECT_ROOT / "outputs" / "dashboard")

## 1. Preparación de datos embebidos

Dos bloques de datos, ambos reducidos en tamaño para que el HTML final
sea manejable (unos pocos MB, no cientos):

1. **Geometría de cantones simplificada** (tolerancia 0.002° ≈ 200 m —
   suficiente para un mapa nacional, reduce el tamaño ~3x sin alterar
   la lectura visual) + los indicadores ya calculados (EFCI y
   sub-índices).
2. **Eventos agregados** por año, mes, provincia, cantón, categoría
   ICEN y tipo de peligro (no evento por evento — 21,743 filas
   individuales serían innecesariamente pesadas para lo que el
   dashboard necesita mostrar: sumas agregadas, no el detalle
   registro-a-registro).

**English:** Prepare the data embedded in the page.

In [ ]:
# Read the master canton dataset
master = gpd.read_file(processed_dir / "mapa_master_canton.gpkg")
# Work on a copy
master_simple = master.copy()
# Simplify geometries (about 200 m) to keep the HTML light
master_simple["geometry"] = master_simple.geometry.simplify(0.002, preserve_topology=True)
# Keep only the columns the dashboard needs
master_simple = master_simple[[
    "canton_code", "canton_name_official", "province_name_official",
    "efci", "clase_efci", "severity_index", "recurrence_index",
    "persistence_index", "enso_magnitude_mean", "gi_clase_efci", "geometry",
]]
# Convert to GeoJSON to embed it in the page
geojson_cantones = json.loads(master_simple.to_json())
# Log the GeoJSON size
logger.info("GeoJSON de cantones preparado: %d features, %.2f MB",
            len(geojson_cantones["features"]), len(json.dumps(geojson_cantones)) / 1e6)

In [ ]:
# Read the georeferenced events
he = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")
# Official canton and province names
canton_names = gpd.read_file(processed_dir / "adm2_canton.geojson")[["canton_code", "canton_name_official"]]
# Canton codes as strings
canton_names["canton_code"] = canton_names["canton_code"].astype(str)
# Canton codes as strings
he["canton_code"] = he["canton_code"].astype(str)
# Attach official names to each event
he = he.merge(canton_names, on="canton_code", how="left")

# Impact variables offered in the dashboard (column: label)
VARIABLES_IMPACTO = {
    "n_eventos": "Nº de Eventos",
    "affected_direct": "Afectados Directos",
    "affected_indirect": "Afectados Indirectos",
    "deaths": "Fallecidos",
    "houses_affected": "Viviendas Afectadas",
    "houses_destroyed": "Viviendas Destruidas",
    "crops_affected_ha": "Cultivos Afectados (ha)",
    "livestock_affected": "Ganado Afectado",
    "livestock_deaths": "Muertes de Ganado",
}

# Aggregate events by year, month, canton, ENSO category and hazard
agg = he.groupby([
    "year", "month", "province_name_official", "canton_code", "canton_name_official",
    "icen_category", "hazard_type",
]).agg(
    n_eventos=("event_id", "count"),
    affected_direct=("affected_direct", "sum"),
    affected_indirect=("affected_indirect", "sum"),
    deaths=("deaths", "sum"),
    houses_affected=("houses_affected", "sum"),
    houses_destroyed=("houses_destroyed", "sum"),
    crops_affected_ha=("crops_affected_ha", "sum"),
    livestock_affected=("livestock_affected", "sum"),
    livestock_deaths=("livestock_deaths", "sum"),
).reset_index()

# Nombres de columna cortos para reducir el peso del JSON embebido
# Short column names to reduce the HTML size
agg = agg.rename(columns={
    "year": "y", "month": "m", "province_name_official": "prov",
    "canton_code": "cc", "canton_name_official": "cn",
    "icen_category": "enso", "hazard_type": "hz",
})
# Convert to records for the page
eventos_json = agg.to_dict(orient="records")
# Log the aggregated size
logger.info("Eventos agregados: %d filas, %.2f MB", len(eventos_json), len(json.dumps(eventos_json)) / 1e6)

# Check the aggregation keys have no nulls
assert agg[["y", "m", "prov", "cc", "enso", "hz"]].isna().sum().sum() == 0 or agg["prov"].isna().sum() <= 40, \
    "revisar nulos en las llaves de agregación (se tolera el pequeño remanente sin provincia geométrica de Fase 2 §4.4)"

## 2. Listas de opciones para los filtros (generadas de los datos, no hardcodeadas)

**English:** Filter option lists generated from the data (not hard-coded).

In [ ]:
# Filter options generated from the data, not hard-coded
PROVINCIAS = sorted(master["province_name_official"].unique().tolist())
# ENSO categories present in the data
CATEGORIAS_ENSO = sorted(he["icen_category"].dropna().unique().tolist())
# Years present in the data
ANIOS = sorted(he["year"].unique().tolist())
# Calendar months
MESES = list(range(1, 13))
# Month labels
MES_NOMBRES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

logger.info("Opciones de filtro: %d provincias, %d categorías ENSO, años %d-%d",
            len(PROVINCIAS), len(CATEGORIAS_ENSO), min(ANIOS), max(ANIOS))

## 3. Plantilla HTML/JS

El archivo es un único HTML con Leaflet (mapa) y Plotly.js (gráfico) vía
CDN — igual que cualquier dashboard estático descargable — más
JavaScript propio (sin dependencias adicionales) para el filtrado y la
tabla. Los datos de las Secciones 1-2 se insertan como JSON embebido
dentro del propio archivo.

**English:** HTML/JavaScript template.

In [ ]:
# HTML + JavaScript template; the data placeholders (__X__) are replaced in the assembly step
HTML_TEMPLATE = '''<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>EFCI Ecuador — Dashboard de Cronicidad de Inundaciones</title>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  body { font-family: Arial, Helvetica, sans-serif; margin: 0; background: #f5f6f8; color: #222; }
  header { background: #1f4e78; color: white; padding: 14px 24px; }
  header h1 { margin: 0; font-size: 20px; }
  header p { margin: 4px 0 0; font-size: 12px; color: #cfe0ee; }
  .layout { display: grid; grid-template-columns: 260px 1fr; gap: 16px; padding: 16px; }
  .filtros { background: white; border-radius: 8px; padding: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.15); height: fit-content; }
  .filtros h3 { margin-top: 0; font-size: 14px; color: #1f4e78; }
  .filtros label { display: block; font-size: 12px; font-weight: bold; margin-top: 12px; margin-bottom: 4px; }
  .filtros select { width: 100%; padding: 6px; font-size: 12px; border-radius: 4px; border: 1px solid #ccc; }
  .filtros button { margin-top: 16px; width: 100%; padding: 8px; background: #1f4e78; color: white; border: none; border-radius: 4px; cursor: pointer; font-size: 12px; }
  .filtros button:hover { background: #163a5a; }
  .panel { background: white; border-radius: 8px; padding: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.15); margin-bottom: 16px; }
  .panel h3 { margin-top: 0; font-size: 14px; color: #1f4e78; }
  #mapa { height: 480px; border-radius: 6px; }
  #grafico { height: 320px; }
  table { width: 100%; border-collapse: collapse; font-size: 12px; }
  th, td { text-align: left; padding: 6px 8px; border-bottom: 1px solid #eee; }
  th { background: #f0f4f8; color: #1f4e78; cursor: pointer; }
  .resumen { display: flex; gap: 12px; margin-bottom: 16px; }
  .tarjeta { flex: 1; background: white; border-radius: 8px; padding: 12px 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.15); text-align: center; }
  .tarjeta .valor { font-size: 22px; font-weight: bold; color: #1f4e78; }
  .tarjeta .etiqueta { font-size: 11px; color: #666; }
  footer { text-align: center; font-size: 11px; color: #888; padding: 12px; }
</style>
</head>
<body>
<header>
  <h1>EFCI Ecuador — Dashboard de Cronicidad de Inundaciones por El Niño</h1>
  <p>FAO — Oficina de Emergencias y Resiliencia (OER) · 2015-2025 · Generado 2026-08-06</p>
</header>
<div class="layout">
  <div class="filtros">
    <h3>Filtros</h3>
    <label for="f-provincia">Provincia</label>
    <select id="f-provincia"><option value="">Todas</option></select>

    <label for="f-enso">Categoría ENSO (ICEN)</label>
    <select id="f-enso"><option value="">Todas</option></select>

    <label for="f-anio-desde">Año (desde)</label>
    <select id="f-anio-desde"></select>

    <label for="f-anio-hasta">Año (hasta)</label>
    <select id="f-anio-hasta"></select>

    <label for="f-mes">Mes</label>
    <select id="f-mes"><option value="">Todos</option></select>

    <label for="f-variable">Variable de Impacto</label>
    <select id="f-variable"></select>

    <button onclick="aplicarFiltros()">Aplicar filtros</button>
    <button onclick="resetFiltros()" style="background:#888;margin-top:8px;">Restablecer</button>
  </div>

  <div>
    <div class="resumen" id="resumen-tarjetas"></div>

    <div class="panel">
      <h3>Mapa — cantones coloreados según la variable de impacto seleccionada (dentro del filtro actual)</h3>
      <div id="mapa"></div>
    </div>

    <div class="panel">
      <h3>Tendencia anual — variable seleccionada, dentro del filtro actual</h3>
      <div id="grafico"></div>
    </div>

    <div class="panel">
      <h3>Ranking de cantones — variable seleccionada, dentro del filtro actual (top 20)</h3>
      <table id="tabla-ranking">
        <thead><tr><th>#</th><th>Cantón</th><th>Provincia</th><th>Valor</th></tr></thead>
        <tbody></tbody>
      </table>
    </div>
  </div>
</div>
<footer>EFCI Ecuador · Jerson Achicanoy (FAO OER) · Metodología completa en docs/methodology · Este archivo no se actualiza automáticamente, regenerar con Step10d tras nuevos datos.</footer>

<script>
const GEOJSON_CANTONES = __GEOJSON_CANTONES__;
const EVENTOS = __EVENTOS__;
const PROVINCIAS = __PROVINCIAS__;
const CATEGORIAS_ENSO = __CATEGORIAS_ENSO__;
const ANIOS = __ANIOS__;
const MESES = __MESES__;
const MES_NOMBRES = __MES_NOMBRES__;
const VARIABLES = __VARIABLES__;  // {clave: etiqueta}

// --- Poblar selects ---
function poblarSelect(id, opciones, formatear) {
  const sel = document.getElementById(id);
  opciones.forEach(op => {
    const o = document.createElement("option");
    o.value = op;
    o.textContent = formatear ? formatear(op) : op;
    sel.appendChild(o);
  });
}
poblarSelect("f-provincia", PROVINCIAS);
poblarSelect("f-enso", CATEGORIAS_ENSO);
poblarSelect("f-anio-desde", ANIOS);
poblarSelect("f-anio-hasta", ANIOS);
poblarSelect("f-mes", MESES, m => MES_NOMBRES[m - 1]);
poblarSelect("f-variable", Object.keys(VARIABLES), k => VARIABLES[k]);
document.getElementById("f-anio-desde").value = ANIOS[0];
document.getElementById("f-anio-hasta").value = ANIOS[ANIOS.length - 1];
document.getElementById("f-variable").value = "n_eventos";

// --- Mapa base ---
const mapa = L.map("mapa").setView([-1.5, -78.5], 6);
L.tileLayer("https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png", { attribution: "OpenStreetMap" }).addTo(mapa);
let capaCantones = null;

function colorEscala(valor, min, max) {
  if (max === min) return "#fee5d9";
  const t = (valor - min) / (max - min);
  const paleta = ["#fee5d9", "#fcae91", "#fb6a4a", "#de2d26", "#a50f15"];
  const idx = Math.min(4, Math.floor(t * 5));
  return paleta[idx];
}

function filtrarEventos() {
  const prov = document.getElementById("f-provincia").value;
  const enso = document.getElementById("f-enso").value;
  const anioDesde = parseInt(document.getElementById("f-anio-desde").value);
  const anioHasta = parseInt(document.getElementById("f-anio-hasta").value);
  const mes = document.getElementById("f-mes").value;

  return EVENTOS.filter(e => {
    if (prov && e.prov !== prov) return false;
    if (enso && e.enso !== enso) return false;
    if (e.y < anioDesde || e.y > anioHasta) return false;
    if (mes && e.m !== parseInt(mes)) return false;
    return true;
  });
}

function agregarPorCanton(eventosFiltrados, variable) {
  const totales = {};
  eventosFiltrados.forEach(e => {
    const key = e.cc;
    if (!totales[key]) totales[key] = { nombre: e.cn, provincia: e.prov, valor: 0 };
    totales[key].valor += (e[variable] || 0);
  });
  return totales;
}

function aplicarFiltros() {
  const variable = document.getElementById("f-variable").value;
  const eventosFiltrados = filtrarEventos();
  const totalesCanton = agregarPorCanton(eventosFiltrados, variable);

  // --- Tarjetas resumen ---
  const totalGeneral = Object.values(totalesCanton).reduce((s, v) => s + v.valor, 0);
  const nCantonesAfectados = Object.keys(totalesCanton).length;
  document.getElementById("resumen-tarjetas").innerHTML = `
    <div class="tarjeta"><div class="valor">${eventosFiltrados.length ? eventosFiltrados.reduce((s,e)=>s+e.n_eventos,0) : 0}</div><div class="etiqueta">Eventos en el filtro actual</div></div>
    <div class="tarjeta"><div class="valor">${totalGeneral.toLocaleString()}</div><div class="etiqueta">${VARIABLES[variable]} (total)</div></div>
    <div class="tarjeta"><div class="valor">${nCantonesAfectados}</div><div class="etiqueta">Cantones con registros en el filtro</div></div>
  `;

  // --- Mapa ---
  const valores = Object.values(totalesCanton).map(v => v.valor);
  const min = valores.length ? Math.min(...valores) : 0;
  const max = valores.length ? Math.max(...valores) : 1;

  if (capaCantones) mapa.removeLayer(capaCantones);
  capaCantones = L.geoJSON(GEOJSON_CANTONES, {
    style: feature => {
      const t = totalesCanton[feature.properties.canton_code];
      return {
        fillColor: t ? colorEscala(t.valor, min, max) : "#e0e0e0",
        weight: 0.5, color: "white", fillOpacity: 0.8,
      };
    },
    onEachFeature: (feature, layer) => {
      const t = totalesCanton[feature.properties.canton_code];
      const valorTexto = t ? t.valor.toLocaleString() : "0";
      layer.bindTooltip(`<b>${feature.properties.canton_name_official}</b><br>${feature.properties.province_name_official}<br>${VARIABLES[variable]}: ${valorTexto}<br>EFCI: ${feature.properties.efci.toFixed(3)} (${feature.properties.clase_efci})`);
    },
  }).addTo(mapa);

  // --- Gráfico de tendencia anual ---
  const porAnio = {};
  eventosFiltrados.forEach(e => { porAnio[e.y] = (porAnio[e.y] || 0) + (e[variable] || 0); });
  const anios = Object.keys(porAnio).sort();
  Plotly.newPlot("grafico", [{
    x: anios, y: anios.map(a => porAnio[a]), type: "bar", marker: { color: "#1f4e78" },
  }], {
    margin: { t: 10, r: 10, b: 40, l: 50 },
    yaxis: { title: VARIABLES[variable] },
    xaxis: { title: "Año" },
  }, { displayModeBar: false });

  // --- Tabla ranking ---
  const filas = Object.values(totalesCanton).sort((a, b) => b.valor - a.valor).slice(0, 20);
  const tbody = document.querySelector("#tabla-ranking tbody");
  tbody.innerHTML = filas.map((f, i) => `<tr><td>${i + 1}</td><td>${f.nombre}</td><td>${f.provincia}</td><td>${f.valor.toLocaleString()}</td></tr>`).join("");
}

function resetFiltros() {
  document.getElementById("f-provincia").value = "";
  document.getElementById("f-enso").value = "";
  document.getElementById("f-anio-desde").value = ANIOS[0];
  document.getElementById("f-anio-hasta").value = ANIOS[ANIOS.length - 1];
  document.getElementById("f-mes").value = "";
  document.getElementById("f-variable").value = "n_eventos";
  aplicarFiltros();
}

aplicarFiltros();
</script>
</body>
</html>
'''
logger.info("Plantilla HTML preparada (%d caracteres antes de insertar datos)", len(HTML_TEMPLATE))

## 4. Ensamblado final e inyección de datos

**English:** Final assembly and data injection.

In [ ]:
# Start from the template
html_final = HTML_TEMPLATE
# Inject the canton GeoJSON
html_final = html_final.replace("__GEOJSON_CANTONES__", json.dumps(geojson_cantones))
# Inject the aggregated events
html_final = html_final.replace("__EVENTOS__", json.dumps(eventos_json))
# Inject the filter options
html_final = html_final.replace("__PROVINCIAS__", json.dumps(PROVINCIAS))
html_final = html_final.replace("__CATEGORIAS_ENSO__", json.dumps(CATEGORIAS_ENSO))
html_final = html_final.replace("__ANIOS__", json.dumps(ANIOS))
html_final = html_final.replace("__MESES__", json.dumps(MESES))
html_final = html_final.replace("__MES_NOMBRES__", json.dumps(MES_NOMBRES))
html_final = html_final.replace("__VARIABLES__", json.dumps(VARIABLES_IMPACTO))

# Output file of the dashboard
dashboard_path = dashboard_dir / "EFCI_Ecuador_Dashboard.html"
# Write the self-contained HTML
dashboard_path.write_text(html_final, encoding="utf-8")
logger.info("Dashboard HTML exportado: %s (%.2f MB)", dashboard_path, dashboard_path.stat().st_size / 1e6)
print(f"Dashboard exportado: {dashboard_path} ({dashboard_path.stat().st_size / 1e6:.2f} MB)")

## 5. Validación

**English:** Validation.

In [ ]:
# Check the file was written and is not trivially small
assert dashboard_path.exists() and dashboard_path.stat().st_size > 1_000_000, "el dashboard parece incompleto (muy pequeño)"
# Read it back
contenido = dashboard_path.read_text(encoding="utf-8")
# Check every placeholder was replaced
for marcador in ["__GEOJSON_CANTONES__", "__EVENTOS__", "__PROVINCIAS__"]:
    assert marcador not in contenido, f"placeholder '{marcador}' no fue reemplazado — el HTML quedaría roto"
# Check the map and chart libraries are referenced
assert "leaflet" in contenido.lower() and "plotly" in contenido.lower(), "faltan las librerías de mapa/gráfico"
logger.info("Fase 10d — validación OK.")
print("Fase 10d completa.")